# Preprocessing

Этот notebook выносит всю базовую инициализацию в отдельный подготовительный слой. Его цель — один раз прозрачно собрать основные панели и проверки, чтобы downstream notebooks не начинались с длинного блока скрытых setup assumptions.

## Что здесь происходит

Здесь мы последовательно:

- поднимаем окружение и helper-функции;
- фиксируем конфигурацию и явные assumptions;
- загружаем исходные рынки и probability history;
- строим основные панели `terminal` и `repricing`;
- делаем sanity checks, coverage checks и показываем ключевые колонки.

Этот notebook должен читаться как reproducible preprocessing protocol, а не как технический хвост paper notebook'а.

## Импорты и окружение

Стандартный stack и локальные benchmark helpers. Как и в основном notebook, Polymarket `crypto` domain здесь исключён из анализа, а `BTC/ETH` используются только как внешние сигналы.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from polymarket_research import PolymarketDataset
from polymarket_research.research.covariate_utils import (
    load_external_covariates,
    pivot_covariates_to_wide,
)
from polymarket_research.research.dataset_utils import prepare_resolved_markets
from polymarket_research.utils import setup_root

REPO_ROOT = setup_root()


## Конфигурация и helper-функции

Ниже задаются все важные параметры preprocessing stage. У helper-функций оставлены короткие docstring'и, чтобы из ячейки было понятно, за что отвечает каждая часть.

In [2]:
DB_PATH = DEFAULT_DB_PATH
DOMAINS = ('politics', 'geopolitics', 'technology', 'finance_economy')
MAX_MARKETS_PER_DOMAIN = 120
MIN_PROBABILITY_ROWS = 288

TERMINAL_HORIZONS = (24, 72, 168)
REPRICING_FUTURE_HOURS = 24
REPRICING_LOOKBACK_HOURS = 24
REPRICING_SAMPLE_EVERY_HOURS = 12
REPRICING_MOVE_THRESHOLD = 0.15

SHOCK_Z_THRESHOLD = 2.0
SHOCK_STD_WINDOW = 288
EXTERNAL_PATH = REPO_ROOT / 'cached_data' / 'external_covariates'

RETRIEVAL_MAX_MARKETS_PER_DOMAIN = 90
TOP_K = 5

def parse_listish(value):
    """Parse a stored tag/list field into a clean Python list of strings."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    if '|' in text:
        parts = text.split('|')
    elif ',' in text:
        parts = text.split(',')
    else:
        parts = [text]
    return [part.strip() for part in parts if part.strip()]

def normalize_text(value: str) -> str:
    """Normalize free text so simple lexical matching is more stable."""
    text = str(value or '').lower()
    text = re.sub(r'[^a-z0-9\s]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_family_id(question: str, domain: str, tags) -> str:
    """Build a weak family identifier for grouping related markets."""
    norm_q = normalize_text(question)
    norm_tags = [normalize_text(tag) for tag in parse_listish(tags)]
    tokens = [tok for tok in norm_q.split() if tok not in {'will', 'the', 'a', 'an', 'be', 'is', 'are', 'to', 'of', 'by', 'in'}]
    key = ' '.join(tokens[:6]) if tokens else norm_q[:48]
    tag_key = '|'.join(sorted(norm_tags[:3]))
    return f"{domain}::{tag_key}::{key}".strip(':')

def tag_jaccard(tags_a, tags_b):
    """Compute simple tag overlap between two markets."""
    a = set(parse_listish(tags_a))
    b = set(parse_listish(tags_b))
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def clipped(p):
    """Clip probabilities away from 0 and 1 for stable metrics."""
    return np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)

def safe_auc(y_true, score):
    """Return ROC-AUC when both classes are present, else NaN."""
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, score)

def safe_ap(y_true, score):
    """Return average precision when both classes are present, else NaN."""
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, score)

def latest_context_snapshot(probabilities_df: pd.DataFrame, market_id: str, cutoff: pd.Timestamp):
    """Fetch the latest probability snapshot for a market before a cutoff."""
    panel = probabilities_df.loc[(probabilities_df['market_id'] == market_id) & (probabilities_df['timestamp_utc'] <= cutoff)]
    if panel.empty:
        return None
    return panel.iloc[-1]

def build_family_context_features(dataset: pd.DataFrame, probabilities_df: pd.DataFrame, market_meta: pd.DataFrame) -> pd.DataFrame:
    """Aggregate same-family market probabilities into simple context features."""
    family_map = market_meta.groupby('family_id')['market_id'].apply(list).to_dict()
    rows = []
    for row in dataset[['market_id', 'cutoff_timestamp_utc', 'market_price_baseline']].itertuples(index=False):
        family_id = market_meta.loc[market_meta['market_id'] == row.market_id, 'family_id'].iloc[0]
        related_ids = [mid for mid in family_map.get(family_id, []) if mid != row.market_id]
        related_probs = []
        for related_id in related_ids:
            snap = latest_context_snapshot(probabilities_df, related_id, row.cutoff_timestamp_utc)
            if snap is None:
                continue
            related_probs.append(float(snap['yes_probability']))
        rows.append({
            'market_id': row.market_id,
            'cutoff_timestamp_utc': row.cutoff_timestamp_utc,
            'family_related_count': float(len(related_probs)),
            'family_prob_mean': float(np.mean(related_probs)) if related_probs else np.nan,
            'family_prob_gap': float(np.max(related_probs) - np.min(related_probs)) if related_probs else np.nan,
            'family_vs_market_gap': float(abs(np.mean(related_probs) - row.market_price_baseline)) if related_probs else np.nan,
        })
    return pd.DataFrame(rows)

def build_shock_table(path: Path, z_threshold: float = 2.0, std_window: int = 288) -> pd.DataFrame:
    """Convert external covariates into returns, z-scores, and shock flags."""
    covariates = load_external_covariates(path)
    wide = pivot_covariates_to_wide(covariates, value_col='value').sort_values('timestamp_utc').reset_index(drop=True)
    out = wide[['timestamp_utc']].copy()
    value_cols = [col for col in wide.columns if col != 'timestamp_utc']
    for col in value_cols:
        series = pd.to_numeric(wide[col], errors='coerce')
        ret = series.pct_change()
        sigma = ret.rolling(std_window, min_periods=max(24, std_window // 6)).std()
        z = ret / sigma.replace(0.0, np.nan)
        out[f'{col}_ret'] = ret
        out[f'{col}_z'] = z
        out[f'{col}_shock'] = (z.abs() >= z_threshold).astype(float)
    out['any_external_shock'] = out[[col for col in out.columns if col.endswith('_shock')]].max(axis=1)
    return out


## Явные assumptions и ограничения текущего preprocessing protocol

- `DOMAINS = politics, geopolitics, technology, finance_economy`
  Ограничиваемся полноценно загруженными и относительно чистыми доменами.
- `MAX_MARKETS_PER_DOMAIN = 120`
  Это вычислительный компромисс: держим preprocessing стабильным и сопоставимым по доменам.
- `MIN_PROBABILITY_ROWS = 288`
  Отсекаем совсем короткие и плохо наблюдаемые рынки.
- `TERMINAL_HORIZONS = 24, 72, 168`
  Фиксируем интерпретируемые горизонты до resolution.
- `REPRICING_FUTURE_HOURS = 24` и `REPRICING_MOVE_THRESHOLD = 0.15`
  Именно так operationally определяется large repricing.
- `REPRICING_SAMPLE_EVERY_HOURS = 12`
  Снижаем автокорреляцию между почти одинаковыми snapshot'ами.
- `SHOCK_Z_THRESHOLD = 2.0`
  BTC/ETH shocks задаются через rolling z-score и нужны как внешний descriptive signal.

## Сбор основных панелей

Это ключевая инициализация всего проекта. Здесь создаются объекты, на которых уже работают terminal, trust, repricing и retrieval analyses. Важный момент: сначала загружаются сырые рынки и probability histories, а уже потом поверх них строятся task-specific datasets.

In [3]:
DATASET_ARTEFACT_DIR = REPO_ROOT / 'research_notebooks' / 'running_artefacts'

dataset = PolymarketDataset.from_parquet(DATASET_ARTEFACT_DIR)
markets = dataset.markets.copy()
probabilities = dataset.probabilities.copy()

prepared_markets = prepare_resolved_markets(markets)
prepared_markets = prepared_markets[prepared_markets['domain'].isin(DOMAINS)].copy()


Terminal rows: 966 Markets: 342
Repricing rows: 80755 Markets: 453
Domains used: politics, geopolitics, technology, finance_economy
Polymarket crypto domain is excluded by design; BTC/ETH are kept only as external signals.


## Какие объекты инициализированы

Ниже краткий inventory: какие датафреймы реально готовы после preprocessing stage и какого они размера. Это удобно, чтобы downstream notebook мог сразу сказать, чем именно он пользуется.

In [4]:

preprocessing_objects = {
    'markets': markets,
    'probabilities': probabilities,
    'terminal': terminal,
    'repricing': repricing,
    'shock_table': shock_table,
}

print('Initialized objects:')
for name, obj in preprocessing_objects.items():
    shape = getattr(obj, 'shape', None)
    if shape is None:
        print(f'- {name}: type={type(obj).__name__}')
    else:
        print(f'- {name}: shape={shape}')


Initialized objects:
- markets: shape=(480, 18)
- probabilities: shape=(12310513, 7)
- terminal: shape=(966, 52)
- repricing: shape=(80755, 37)
- shock_table: shape=(131904, 8)


## Coverage и ключевые колонки

Эта секция показывает, насколько хорошо покрыты домены и где именно находятся основные источники signal. Особенно важно смотреть на `rows`, `markets`, `repricing_rate`, `market_abs_error`, `future_move`.

In [5]:

markets_summary = markets.assign(market_age_days=(markets['end_date'] - markets['created_at']).dt.total_seconds() / 86400.0).groupby('primary_domain', dropna=False).agg(
    markets=('market_id', 'nunique'),
    mean_market_age_days=('market_age_days', 'mean'),
    mean_probability_rows=('probability_rows', 'mean'),
    mean_trade_rows=('trade_rows', 'mean'),
).reset_index().sort_values('markets', ascending=False)

display(markets_summary)

terminal_summary = terminal.groupby(['primary_domain', 'horizon_name'], dropna=False).agg(
    rows=('market_id', 'size'),
    markets=('market_id', 'nunique'),
    positive_rate=('target', 'mean'),
    mean_abs_error=('market_abs_error', 'mean'),
).reset_index()
repricing_summary = repricing.groupby('primary_domain', dropna=False).agg(
    rows=('market_id', 'size'),
    markets=('market_id', 'nunique'),
    repricing_rate=('target', 'mean'),
    mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
).reset_index()
display(terminal_summary)
display(repricing_summary)

key_columns_guide = pd.DataFrame([
    {'panel': 'terminal', 'column': 'market_price_baseline', 'meaning': 'текущая рыночная вероятность на момент cutoff; это главный baseline'},
    {'panel': 'terminal', 'column': 'market_abs_error', 'meaning': 'насколько текущая вероятность далека от финального исхода'},
    {'panel': 'terminal', 'column': 'family_prob_gap', 'meaning': 'разброс вероятностей среди родственных рынков'},
    {'panel': 'repricing', 'column': 'future_move', 'meaning': 'сдвиг вероятности за следующие 24 часа'},
    {'panel': 'repricing', 'column': 'recent_volatility', 'meaning': 'локальная нестабильность рынка до текущего момента'},
    {'panel': 'repricing', 'column': 'btc_or_eth_shock', 'meaning': 'внешний shock-флаг по BTC/ETH на момент snapshot'},
])
display(key_columns_guide)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=terminal_summary, x='rows', y='primary_domain', hue='horizon_name', ax=axes[0], palette='crest')
axes[0].set_title('Покрытие terminal panel по доменам и горизонтам')
axes[0].set_xlabel('Число строк')
axes[0].set_ylabel('')

sns.barplot(data=repricing_summary, x='repricing_rate', y='primary_domain', ax=axes[1], palette='flare')
axes[1].set_title('Доля large repricing по доменам')
axes[1].set_xlabel('Частота repricing')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

terminal_key_columns = terminal[[
    'market_id', 'primary_domain', 'horizon_name', 'market_price_baseline', 'current_yes_probability', 'market_abs_error', 'hours_to_resolution', 'family_related_count', 'family_prob_gap'
]].head(8)
repricing_key_columns = repricing[[
    'market_id', 'primary_domain', 'timestamp_utc', 'current_yes_probability', 'future_move', 'target', 'recent_volatility', 'confidence_margin', 'btc_or_eth_shock'
]].head(8)
display(terminal_key_columns)
display(repricing_key_columns)


## Sanity checks

Перед тем как идти в modelling, полезно убедиться, что базовые объекты не разваливаются по пропускам и что временные окна выглядят разумно.

- `missing_table` показывает долю пропусков в ключевых колонках;
- `latest_windows` даёт грубую temporal coverage проверку.

In [6]:

missing_table = pd.DataFrame([
    {
        'object': 'markets',
        'rows': len(markets),
        'key_missing_share': markets[['market_id', 'question', 'end_date']].isna().mean().mean(),
    },
    {
        'object': 'probabilities',
        'rows': len(probabilities),
        'key_missing_share': probabilities[['market_id', 'timestamp_utc', 'yes_probability']].isna().mean().mean(),
    },
    {
        'object': 'terminal',
        'rows': len(terminal),
        'key_missing_share': terminal[['market_id', 'cutoff_timestamp_utc', 'market_price_baseline', 'target']].isna().mean().mean(),
    },
    {
        'object': 'repricing',
        'rows': len(repricing),
        'key_missing_share': repricing[['market_id', 'timestamp_utc', 'current_yes_probability', 'target']].isna().mean().mean(),
    },
]).sort_values('rows', ascending=False)
display(missing_table)

latest_windows = pd.DataFrame([
    {
        'object': 'markets',
        'min_time': markets['created_at'].min(),
        'max_time': markets['end_date'].max(),
    },
    {
        'object': 'probabilities',
        'min_time': probabilities['timestamp_utc'].min(),
        'max_time': probabilities['timestamp_utc'].max(),
    },
    {
        'object': 'terminal_cutoffs',
        'min_time': terminal['cutoff_timestamp_utc'].min(),
        'max_time': terminal['cutoff_timestamp_utc'].max(),
    },
    {
        'object': 'repricing_snapshots',
        'min_time': repricing['timestamp_utc'].min(),
        'max_time': repricing['timestamp_utc'].max(),
    },
])
display(latest_windows)


## Примеры строк из основных панелей

В конце полезно глазами посмотреть несколько первых строк из `terminal` и `repricing`. Это помогает быстро вспомнить, какие поля дальше будут центральными в моделях.

In [7]:

terminal_sample = terminal[[
    'market_id', 'primary_domain', 'horizon_name', 'cutoff_timestamp_utc', 'market_price_baseline', 'target', 'confidence_margin', 'family_related_count', 'family_prob_gap'
]].head(10)
repricing_sample = repricing[[
    'market_id', 'primary_domain', 'timestamp_utc', 'current_yes_probability', 'future_move', 'target', 'recent_volatility', 'confidence_margin', 'btc_or_eth_shock'
]].head(10)

display(terminal_sample)
display(repricing_sample)
